In [1]:
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
import io
import csv
import numpy as np
import pandas as pd
import seaborn as sns
import importlib
import os
import functools
import itertools
import torch

import torch.nn as nn
# import tensorflow as tf
# import tensorflow_datasets as tfds
# import tensorflow_gan as tfgan
# import tqdm
import io
import inspect
sns.set(font_scale=2)
sns.set(style="whitegrid")

import models
from models import utils as mutils
from models import ncsnv2
from models import ncsnpp
from models import ddpm as ddpm_model
from models import layerspp
from models import layers
from models import normalization

#from configs.ncsnpp import cifar10_continuous_ve as configs
# from configs.ddpm import cifar10_continuous_vp as configs
# config = configs.get_config()



In [ ]:
from configs.ve import MultiRIR_ncsnpp_continuous as configs
config = configs.get_config()

# checkpoint = torch.load('exp/ddpm_continuous_vp.pth')

#score_model = ncsnpp.NCSNpp(config)
# score_model = ddpm_model.DDPM(config)
score_model = ncsnpp.NCSNpp(config)
# score_model.load_state_dict(checkpoint)
score_model = score_model.eval()
x = torch.ones(8, 1, 1024, 32)
temb = torch.tensor([1] * 8)
y = torch.ones(8, 1, 1024, 32)
with torch.no_grad():
  score = score_model(x, temb, y)

ModuleList(
  (0): GaussianFourierProjection()
  (1): Linear(in_features=256, out_features=512, bias=True)
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): Conv2d(64, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (4): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5-6): 2 x ResnetBlockBigGANpp(
    (GroupNorm_0): GroupNorm(32, 128, eps=1e-06, affine=True)
    (Conv_0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (Dense_0): Linear(in_features=512, out_features=128, bias=True)
    (GroupNorm_1): GroupNorm(32, 128, eps=1e-06, affine=True)
    (Dropout_0): Dropout(p=0.0, inplace=False)
    (Conv_1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (act): SiLU()
  )
  (7): ResnetBlockBigGANpp(
    (GroupNorm_0): GroupNorm(32, 128, eps=1e-06, affine=True)
    (Conv_0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (Dense_0): Linear(in_features=512, out_features=128

In [3]:
score.shape

torch.Size([8, 1, 1024, 32])

In [2]:
pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html

Looking in links: https://download.pytorch.org/whl/torch_stable.htmlNote: you may need to restart the kernel to use updated packages.

  Using cached https://download.pytorch.org/whl/cu110/torch-1.7.1%2Bcu110-cp38-cp38-win_amd64.whl
  Using cached https://download.pytorch.org/whl/cu110/torchvision-0.8.2%2Bcu110-cp38-cp38-win_amd64.whl
  Using cached https://download.pytorch.org/whl/torchaudio-0.7.2-cp38-none-win_amd64.whl
  Using cached https://files.pythonhosted.org/packages/f2/75/3cb820b2812405fc7feb3d0deb701ef0c3de93dc02597115e00704591bc9/pillow-10.4.0-cp38-cp38-win_amd64.whl


You should consider upgrading via the 'python -m pip install --upgrade pip' command.


In [1]:
pip install -r requirements_jd2.txt


  Using cached absl_py-0.10.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached certifi-2025.1.31-py3-none-any.whl.metadata (2.5 kB)
  Using cached charset_normalizer-3.4.1-cp312-cp312-win_amd64.whl.metadata (36 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached contextlib2-21.6.0-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached contourpy-1.3.1-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached decorator-5.1.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached docker_pycreds-0.4.0-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached filelock-3.17.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached fonttools-4.56.0-cp312-cp312-win_amd64.whl.metadata (103 kB)
  Using cached fsspec-2025.2.0-py3-none-any.whl.metadata (11 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached GitPython-3.1.4


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip freeze > requirements_jd2.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
"""Training and evaluation"""

import run_lib_jd as run_lib
import argparse
import runpy
import logging
import os

def load_config(config_path):
    """
    Charge la configuration depuis un fichier Python.
    Le fichier doit définir une fonction `get_config()` qui retourne un ml_collections.ConfigDict.
    """
    config_module = runpy.run_path(config_path)
    if "get_config" in config_module:
        return config_module["get_config"]()
    elif "config" in config_module:
        return config_module["config"]
    else:
        raise ValueError(f"Le fichier {config_path} ne contient ni 'get_config' ni 'config'.")

def main():
    parser = argparse.ArgumentParser(description="Training and evaluation script.")
    parser.add_argument("--config", type=str, required=True,
                        help="Chemin vers le fichier de configuration Python.")
    parser.add_argument("--workdir", type=str, required=True,
                        help="Répertoire de travail.")
    parser.add_argument("--mode", type=str, choices=["train", "eval"], required=True,
                        help="Mode d'exécution : train ou eval.")
    parser.add_argument("--eval_folder", type=str, default="eval",
                        help="Nom du dossier pour stocker les résultats d'évaluation.")
    args = parser.parse_args()

    # Charger la configuration depuis le fichier Python
    config = load_config(args.config)

    # Création du répertoire de travail
    os.makedirs(args.workdir, exist_ok=True)

    # Configuration du logger pour écrire à la fois sur la console et dans un fichier
    log_file = os.path.join(args.workdir, 'stdout.txt')
    gfile_stream = open(log_file, 'w')
    handler = logging.StreamHandler(gfile_stream)
    formatter = logging.Formatter('%(levelname)s - %(filename)s - %(asctime)s - %(message)s')
    handler.setFormatter(formatter)
    logger = logging.getLogger()
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

    # Exécuter le pipeline en fonction du mode choisi
    if args.mode == "train":
        run_lib.train(config, args.workdir)
    elif args.mode == "eval":
        run_lib.evaluate(config, args.workdir, args.eval_folder)
    else:
        raise ValueError(f"Mode {args.mode} non reconnu.")


In [2]:
import sys
sys.argv = [
    "debug_jd.ipynb",  # nom du script ou notebook
    "--config", "configs/ve/MultiRIR_ncsnpp_continuous.py",
    "--workdir", "exp/ve/MultiRIR_ncsnpp_continuous",
    "--mode", "train",
    "--eval_folder", "eval"
]
main()

perfect_rir.shape torch.Size([2, 32, 1, 1024])
perfect_rir.shape torch.Size([2, 1, 1024, 32])
real_rir.shape torch.Size([2, 32, 1, 1024])
real_rir.shape torch.Size([2, 1, 1024, 32])
ModuleList(
  (0): GaussianFourierProjection()
  (1): Linear(in_features=256, out_features=512, bias=True)
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): Conv2d(64, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (4): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5-6): 2 x ResnetBlockBigGANpp(
    (GroupNorm_0): GroupNorm(32, 128, eps=1e-06, affine=True)
    (Conv_0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (Dense_0): Linear(in_features=512, out_features=128, bias=True)
    (GroupNorm_1): GroupNorm(32, 128, eps=1e-06, affine=True)
    (Dropout_0): Dropout(p=0.0, inplace=False)
    (Conv_1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (act): ELU(alpha=1.0)
  )
  (7): ResnetBlockBigGANpp(
    (Grou

StopIteration: 